# ADAPT — SFT Warmup → GRPO Training Pipeline

**Two-phase training:**
1. **SFT Warmup** — Teach the model correct JSON format using heuristic baseline outputs
2. **GRPO** — Optimise decision quality beyond the heuristic using reinforcement learning

**Why SFT first?** A 1.5B model needs to learn JSON structure before GRPO can optimise the content. Without SFT, ~70% of early GRPO episodes produce invalid JSON → wasted compute + noisy gradients.

**Training mix:** ~50% ADAPT (domain transfer), ~25% AMAN, ~25% DMAN  
**Recommended runtime:** `Runtime → Change runtime type → GPU (T4)`

## 1. Configuration

In [ ]:
from pathlib import Path
import os

REPO_URL   = "https://github.com/GTsingh600/AirX.git"
REPO_DIR   = Path("/content/ATC")
USE_DRIVE  = True
OUTPUT_DIR = (
    Path("/content/drive/MyDrive/atc-adapt")
    if USE_DRIVE else Path("/content/atc-adapt")
)
SFT_DIR    = str(OUTPUT_DIR / "sft-warmup")
GRPO_DIR   = str(OUTPUT_DIR / "grpo-final")

MODEL_NAME      = "Qwen/Qwen2.5-1.5B-Instruct"  # 1.5B fits T4
SFT_EPISODES    = 50     # SFT data episodes (→ ~125 samples)
SFT_EPOCHS      = 3      # SFT training epochs
GRPO_EPISODES   = 100    # GRPO episodes after SFT
LORA_RANK       = 32
N_GENERATIONS   = 2      # GRPO group size (T4 safe)
SEED            = 42

os.environ["WANDB_MODE"]             = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"]       = "1"

print(f"MODEL       = {MODEL_NAME}")
print(f"SFT_DIR     = {SFT_DIR}")
print(f"GRPO_DIR    = {GRPO_DIR}")
print(f"SFT:  {SFT_EPISODES} episodes × {SFT_EPOCHS} epochs")
print(f"GRPO: {GRPO_EPISODES} episodes")

## 2. Mount Drive + Clone Repo

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output ready: {OUTPUT_DIR}")

In [ ]:
import shutil, subprocess, sys, os
from pathlib import Path

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print(f"Working directory: {Path.cwd()}")

## 3. Install Dependencies

In [ ]:
import subprocess, sys

def pip(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

pip("-U", "pip")
pip(
    "unsloth[colab-new]",
    "trl==0.15.2",
    "transformers==4.51.3",
    "datasets>=2.20.0",
    "accelerate>=0.32.0",
    "peft>=0.12.0",
    "bitsandbytes>=0.43.0",
    "matplotlib>=3.9.0",
    "numpy>=1.26.0",
    "openenv-core[core]>=0.2.3",
    "fastapi>=0.111.0",
    "openai>=1.30.0",
    "pydantic>=2.7.0",
    "uvicorn>=0.29.0",
)
print("Done.")

## 4. Sanity Check

In [ ]:
import torch
from training.dataset import build_episode_dataset

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU:  {props.name}  |  VRAM: {props.total_memory/1e9:.1f} GB")

sample = build_episode_dataset(
    n_episodes=4, seed=SEED,
    include_generator=False, include_supervisor=False, include_adapt=True,
    domain_episode_ratio=0.50, long_horizon_ratio=0.0,
)
roles = sorted({r["agent_role"] for r in sample})
print(f"Dataset smoke: {len(sample)} samples | roles: {roles}")
print("3 roles: ADAPT (primary) + AMAN + DMAN (support)")

## 5. Phase 1: SFT Warmup (~15 min on T4)

Teaches the model to output valid JSON in the correct format for each role.  
Training data comes from heuristic baselines — the model learns to **mimic** the heuristic output format.

After SFT, the model should produce valid JSON ~95% of the time (vs ~30% from base model).

In [ ]:
from training.train_sft import train_sft

sft_checkpoint = train_sft(
    model_name  = MODEL_NAME,
    output_dir  = SFT_DIR,
    n_episodes  = SFT_EPISODES,
    lora_rank   = LORA_RANK,
    seed        = SEED,
    num_epochs  = SFT_EPOCHS,
)
print(f"\nSFT checkpoint: {sft_checkpoint}")

## 6. Phase 2: GRPO Training (~1 hour on T4)

Now that the model can produce valid JSON, GRPO optimises the **quality** of decisions.  
Starts from the SFT checkpoint, not the base model.

> **Key difference:** GRPO starting from SFT has ~95% valid JSON (vs ~30% cold start).  
> This means GRPO gets useful gradient signal from episode 1, not episode 50.

In [ ]:
import training.train_grpo as _grpo

os.makedirs(GRPO_DIR, exist_ok=True)

_grpo.train(
    model_name  = sft_checkpoint,   # Start from SFT, not base model!
    output_dir  = GRPO_DIR,
    n_episodes  = GRPO_EPISODES,
    lora_rank   = LORA_RANK,
    seed        = SEED,
    run_eval    = True,
)

## 7. Plot Reward Curves

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

plots_dir = Path(GRPO_DIR) / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

curves_path = Path(GRPO_DIR) / "reward_curves.json"
if not curves_path.exists():
    print("No reward curves found — GRPO may not have completed.")
else:
    with open(curves_path) as f:
        curves = json.load(f)

    COLOURS = {"ADAPT":"#9C27B0", "AMAN":"#2196F3", "DMAN":"#4CAF50", "composite":"#212121"}

    def smooth(v, w=20):
        return [sum(v[max(0,i-w):i+1])/min(w,i+1) for i in range(len(v))]

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle("ADAPT SFT→GRPO — Reward Curves", fontsize=14, fontweight="bold")

    for ax, role in zip(axes.flat, ["ADAPT", "AMAN", "DMAN", "composite"]):
        vals = curves.get(role, [])
        if not vals:
            ax.set_title(f"{role} (no data)"); continue
        col = COLOURS.get(role, "#607D8B")
        ax.plot(vals, alpha=0.2, color=col, lw=0.7)
        ax.plot(smooth(vals), color=col, lw=2.0)
        tail = vals[-max(1,len(vals)//4):]
        fm = sum(tail)/len(tail)
        ax.axhline(fm, color=col, lw=1.0, ls=":")
        ax.set_title(f"{role}  (final={fm:+.3f})", fontweight="bold", color=col)
        ax.set_xlabel("Training call")
        ax.set_ylabel("Reward")
        ax.set_ylim(-1.05, 1.05)
        ax.axhline(0, color="gray", lw=0.5, ls="--")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    p1 = plots_dir / "sft_grpo_curves.png"
    plt.savefig(p1, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {p1}")

## 8. Before / After Comparison

In [ ]:
import json, matplotlib.pyplot as plt, numpy as np
from pathlib import Path

plots_dir = Path(GRPO_DIR) / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

bp = Path(GRPO_DIR) / "baseline_metrics.json"
tp = Path(GRPO_DIR) / "trained_model_metrics.json"

if not bp.exists() or not tp.exists():
    print("Metrics files not found — training may have used --no_eval.")
else:
    with open(bp) as f: baseline = json.load(f)
    with open(tp) as f: trained  = json.load(f)

    metrics = [
        ("mean_composite",   "Composite score"),
        ("mean_aman_reward", "AMAN reward"),
        ("mean_dman_reward", "DMAN reward"),
        ("mean_conflicts",   "Avg conflicts"),
        ("mean_emg_handled", "Emergencies handled"),
    ]
    labels = [m[1] for m in metrics]
    before = [baseline.get(m[0], 0) for m in metrics]
    after  = [trained.get(m[0], 0)  for m in metrics]

    x, w = np.arange(len(labels)), 0.35
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x-w/2, before, w, label="Heuristic Baseline", color="#90A4AE")
    ax.bar(x+w/2, after,  w, label="SFT+GRPO Trained",  color="#9C27B0")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.set_ylabel("Score")
    ax.set_title("Before vs. After — SFT→GRPO Pipeline", fontweight="bold")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    for b, a, xi in zip(before, after, x):
        d = a - b
        ax.text(xi+w/2, max(a,b)+0.02, f"{d:+.3f}", ha="center", fontsize=9,
                color="green" if d > 0 else "red")
    plt.tight_layout()
    p = plots_dir / "before_after_sft_grpo.png"
    plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {p}")

    print("\n=== BEFORE vs AFTER ===")
    for key, label in metrics:
        b, a = baseline.get(key,0), trained.get(key,0)
        d = a - b
        arr = "UP" if d > 0.005 else ("DN" if d < -0.005 else "--")
        print(f"  {label:34s}: {b:6.3f}  ->  {a:6.3f}  ({d:+.3f} {arr})")

## Troubleshooting

| Problem | Fix |
|---|---|
| OOM during SFT | Drop `SFT_EPISODES` to 30 or `LORA_RANK` to 16 |
| OOM during GRPO | Already using N_GENERATIONS=2. Try 1.5B model. |
| SFT loss not decreasing | Normal for 1-2 epochs. If flat after epoch 2, check data. |
| GRPO rewards stuck at 0 after SFT | SFT checkpoint may be corrupted. Check `sft_checkpoint` path. |
| Colab disconnects | Output is on Drive. Resume from SFT checkpoint. |